# FibRec, FibMemo e FibIt (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_java.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `FibRec`, `FibMemo` e `FibIt`: gli algoritmi in Java, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [2]:
%%writefile Fibonacci.java
import java.util.*;

public class Fibonacci {
    static long fibCalls = 0;   // chiamate eseguite

    // fibRec(n)
    static long fibRec(int n) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        return fibRec(n - 1) + fibRec(n - 2);
    }

    // fibMemo(n, memo): memo[k] = -1 significa «non definito»
    static long fibMemo(int n, long[] memo) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        if (memo[n] == -1) {
            memo[n] = fibMemo(n - 1, memo) + fibMemo(n - 2, memo);
        }
        return memo[n];
    }

    // fibIt(n)
    static long fibIt(int n) {
        if (n <= 1) {
            return n;
        }
        long prev = 0;                  // F(i-2)
        long curr = 1;                  // F(i-1)
        int i = 2;
        while (i <= n) {
            long next = prev + curr;
            prev = curr;
            curr = next;
            i = i + 1;
        }
        return curr;
    }

    // Come fibRec, ma conta le chiamate per ogni valore del parametro
    static long fibRecHist(int n, int[] counts) {
        counts[n]++;
        if (n <= 1) {
            return n;
        }
        return fibRecHist(n - 1, counts) + fibRecHist(n - 2, counts);
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] counts = new int[6];
        long f5 = fibRecHist(5, counts);
        System.out.println("FibRec(5) = " + f5);
        StringBuilder line = new StringBuilder();
        int total = 0;
        for (int k = 5; k >= 0; k--) {
            if (k < 5) line.append("  ");
            line.append(k).append(":").append(counts[k]);
            total += counts[k];
        }
        System.out.println("chiamate per ogni valore del parametro: " + line + "   totale " + total);
        System.out.println();
        System.out.printf("%3s %8s %9s %10s %8s %6s  (chiamate, chiamate, chiamate, iterazioni)%n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
        int[] ns = {10, 15, 20, 25, 30};
        for (int n : ns) {
            long f = fibIt(n);
            fibCalls = 0;
            long r1 = fibRec(n);
            long c1 = fibCalls;
            fibCalls = 0;
            long[] memo = new long[n + 1];
            Arrays.fill(memo, -1);
            long r2 = fibMemo(n, memo);
            long c2 = fibCalls;
            boolean ok = (r1 == r2 && r2 == f && c1 == 2 * fibIt(n + 1) - 1 && c2 == 2 * n - 1);
            System.out.printf("%3d %8d %9d %10d %8d %6d  %s%n", n, f, c1, 2 * fibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (int n : new int[]{20, 25, 30}) {
            double t0 = nowMs(); fibRec(n); double t1 = nowMs() - t0;
            t0 = nowMs();
            long sink = 0;
            for (int rep = 0; rep < 1000; rep++) {
                sink += fibIt(n);
            }
            double t2 = (nowMs() - t0) / 1000;
            System.out.printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms%n", n, t1, t2);
        }
    }
}

Writing Fibonacci.java


In [3]:
!javac Fibonacci.java && java Fibonacci

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     0.0367 ms   FibIt   0.000222 ms
n=25  FibRec     0.3892 ms   FibIt   0.000073 ms
n=30  FibRec     4.9251 ms   FibIt   0.000042 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [4]:
%%writefile ProvaFibonacci.java
import java.util.*;

public class ProvaFibonacci {
    public static void main(String[] args) {
        // Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
        int n = 32;
        Fibonacci.fibCalls = 0;
        long r1 = Fibonacci.fibRec(n);
        System.out.println(r1 + " " + Fibonacci.fibCalls);
        Fibonacci.fibCalls = 0;
        long[] memo = new long[n + 1];
        Arrays.fill(memo, -1);
        long r2 = Fibonacci.fibMemo(n, memo);
        System.out.println(r2 + " " + Fibonacci.fibCalls);
    }
}

Writing ProvaFibonacci.java


In [5]:
!javac ProvaFibonacci.java && java ProvaFibonacci

2178309 7049155
2178309 63


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).